In [1]:
import numpy as np
import random
import math

# ============================================================
# Lab 07 – Task #3 : Network Bandwidth Allocation using ABC
# Planning Techniques for Robotics 2025/2026
# ============================================================


# Total bandwidth available (GB → treated as Gbps units)
TOTAL_BW = 750


# Upper bound per client = min(effective_max, TOTAL_BW)
clients = {
    'Client': [1, 2, 3, 4, 5, 6],
    'Tier':   [2, 3, 1, 4, 1, 2],
    'Type':   ['Special','Regular','Special','Regular','Special','Special'],
    'Base_BW':[100, 25, 200, 10, 200, 100],
    'Additional': [0.30, 0.30, 0.30, 0.0, 0.0, 0.22],
}

# Compute effective upper bounds per client
ub_per_client = []
for i in range(6):
    effective = clients['Base_BW'][i] * (1 + clients['Additional'][i])
    ub_per_client.append(min(effective, TOTAL_BW))

priorities = [5 - t for t in clients['Tier']]
# priorities: Client1→3, Client2→2, Client3→4, Client4→1, Client5→4, Client6→3

print("=" * 65)
print("       Lab 07 – Task #3 : ABC – Bandwidth Allocation")
print("=" * 65)
print(f"\nTotal Bandwidth Pool : {TOTAL_BW} Gbps")
print(f"\n{'Client':<10} {'Tier/Type':<14} {'Max BW':>10} {'Priority':>10}")
print("-" * 50)
for i in range(6):
    tier_type = f"{clients['Tier'][i]}/{clients['Type'][i]}"
    print(f"  {i+1:<8} {tier_type:<14} {ub_per_client[i]:>9.1f}  {priorities[i]:>9}")

print()

# ------------------------------------------------------------------
# Networking Class (Artificial Bee Colony)
# ------------------------------------------------------------------
class networking:

    def __init__(self, users_priority, no_of_bees, iterations, limit,
                 penalty=None, lb=None, ub=None):
        """
        users_priority : list of priority weights per client/dimension
        no_of_bees     : colony size
        iterations     : max iterations
        limit          : trial counter threshold (scout trigger)
        penalty        : penalty value for constraint violation
        lb             : lower bound (scalar or array per dimension)
        ub             : upper bound (scalar) – total bandwidth cap
        """
        self.__users_priority = users_priority
        self.__lb = lb           # minimum per-client bandwidth
        self.__ub = ub           # maximum total bandwidth
        self.__pen = penalty     # penalty constant
        self.__ub_per_client = ub_per_client  # per-client upper bounds
        self.__initialize_parameters(no_of_bees, iterations, limit)

    # ----------------------------------------------------------------
    def __initialize_parameters(self, no_of_bees, iterations, limit):
        self.__no_of_bees  = no_of_bees
        self.__iterations  = iterations
        self.__limit       = limit
        self.__dimensions  = len(self.__users_priority)
        self.__dims        = range(self.__dimensions)

        # Initialise population: X_i in [lb, ub_per_client_i]
        # Shape: (no_of_bees, dimensions)
        self.__population = np.array([
            self.__random_solution() for _ in range(self.__no_of_bees)
        ])

        # Evaluate initial population
        self.__objective_vec = np.array([
            self.__objective_function(x) for x in self.__population
        ])
        penalties = np.array([
            self.__penalty(x) for x in self.__population
        ])
        self.__fitness_vec = np.array([
            self.__fitness_function((self.__objective_vec[i], penalties[i]))
            for i in range(self.__no_of_bees)
        ])
        self.__trials = np.zeros(self.__no_of_bees)

    # ----------------------------------------------------------------
    def __random_solution(self):
        """Generate a random feasible solution respecting per-client bounds."""
        sol = np.array([
            np.random.uniform(self.__lb, self.__ub_per_client[d])
            for d in self.__dims
        ])
        # Scale down if total exceeds TOTAL_BW to keep feasibility
        total = sum(sol)
        if total > self.__ub:
            sol = sol * (self.__ub / total)
        return sol

    # ----------------------------------------------------------------
    def __objective_function(self, var):
        """
        Maximise total weighted satisfaction:
        f(x) = Σ priority_i * ln(bandwidth_i)
        """
        obj = [self.__users_priority[x] * math.log(var[x]) for x in self.__dims]
        return sum(obj)

    # ----------------------------------------------------------------
    def __penalty(self, var):
        """
        Returns +1 (penalise) if total bandwidth exceeds TOTAL_BW,
        else -1 (valid solution).
        """
        if sum(var) > self.__ub:
            return 1
        else:
            return -1

    # ----------------------------------------------------------------
    def __fitness_function(self, obj_fun):
        """
        Fitness for MAXIMISATION problem.
        obj_fun = (f(x), penalty_flag)
        If f(x) >= 0 and not penalised → fitness = f(x)
        If f(x) >= 0 and penalised     → fitness = 1 / (f(x) + penalty)
        If f(x) < 0                    → fitness = 1 / (1 + |f(x)|)
        """
        if obj_fun[0] >= 0:
            if obj_fun[1] == 1:          # penalised
                return 1 / (obj_fun[0] + self.__pen)
            else:
                return obj_fun[0]
        else:
            return 1 / (1 + abs(obj_fun[0]))

    # ----------------------------------------------------------------
    def __greedy_selection(self, solutions, objectives, fitness, trial):
        """
        Compare new vs current solution; keep the better one.
        Parameters are TUPLES: (current, new)
        """
        if fitness[1] > fitness[0]:
            return solutions[1], objectives[1], fitness[1], 0
        else:
            return solutions[0], objectives[0], fitness[0], trial + 1

    # ----------------------------------------------------------------
    def __new_solution(self, current_solution, random_partner, random_variable):
        """
        Generate neighbour solution:
        X_new = X + φ*(X - X_partner),  φ ∈ [-1, 1]
        Then clamp to [lb, ub_per_client].
        """
        phi  = np.random.uniform(-1, 1)
        X    = current_solution[random_variable]
        Xp   = random_partner[random_variable]
        Xnew = X + phi * (X - Xp)

        # Clamp to individual bounds
        Xnew = max(self.__lb, min(Xnew, self.__ub_per_client[random_variable]))

        new_solution = current_solution.copy()
        new_solution[random_variable] = Xnew

        new_obj     = self.__objective_function(new_solution)
        new_pen     = self.__penalty(new_solution)
        new_fitness = self.__fitness_function((new_obj, new_pen))

        return new_solution, new_obj, new_fitness

    # ----------------------------------------------------------------
    def __employed_phase(self):
        """
        Each employed bee explores its food source and applies greedy
        selection. OFFLINE update (no in-place modification).
        """
        population    = []
        objective_vec = []
        fitness_vec   = []
        trials        = []

        for ix in range(self.__no_of_bees):
            current_solution = self.__population[ix]
            current_obj      = self.__objective_vec[ix]
            current_fit      = self.__fitness_vec[ix]

            # Pick random partner (different from ix) and random dimension
            partner_pool = [p for p in self.__population if not np.array_equal(p, current_solution)]
            random_partner   = random.choice(partner_pool) if partner_pool else random.choice(self.__population)
            random_variable  = np.random.choice(self.__dims)

            new_solution, new_obj, new_fit = self.__new_solution(
                current_solution, random_partner, random_variable
            )

            selected_sol, selected_obj, selected_fit, trial = self.__greedy_selection(
                (current_solution, new_solution),
                (current_obj,      new_obj),
                (current_fit,      new_fit),
                self.__trials[ix]
            )

            population.append(selected_sol)
            objective_vec.append(selected_obj)
            fitness_vec.append(selected_fit)
            trials.append(trial)

        # Offline update
        self.__population    = np.array(population)
        self.__objective_vec = np.array(objective_vec)
        self.__fitness_vec   = np.array(fitness_vec)
        self.__trials        = np.array(trials)

    # ----------------------------------------------------------------
    def __onlooker_phase(self):
        """
        Onlooker bees select food sources probabilistically (roulette
        wheel) and try to improve them. ONLINE update.
        """
        probabilities  = self.__fitness_vec.copy()
        sum_of_fit     = sum(probabilities)
        probabilities  = np.array(probabilities) / sum_of_fit

        ix = 0
        for _ in range(self.__no_of_bees):
            while True:
                random_number = np.random.random()
                if random_number <= probabilities[ix]:
                    current_solution = self.__population[ix]
                    current_obj      = self.__objective_vec[ix]
                    current_fit      = self.__fitness_vec[ix]

                    partner_pool = [p for p in self.__population if not np.array_equal(p, current_solution)]
                    random_partner  = random.choice(partner_pool) if partner_pool else random.choice(self.__population)
                    random_variable = np.random.choice(self.__dims)

                    new_solution, new_obj, new_fit = self.__new_solution(
                        current_solution, random_partner, random_variable
                    )

                    selected_sol, selected_obj, selected_fit, trial = self.__greedy_selection(
                        (current_solution, new_solution),
                        (current_obj,      new_obj),
                        (current_fit,      new_fit),
                        self.__trials[ix]
                    )

                    # Online update
                    self.__population[ix]    = selected_sol
                    self.__objective_vec[ix] = selected_obj
                    self.__fitness_vec[ix]   = selected_fit
                    self.__trials[ix]        = trial

                    ix = (ix + 1) % self.__no_of_bees
                    break

                ix = (ix + 1) % self.__no_of_bees

    # ----------------------------------------------------------------
    def __scout_phase(self):
        """
        If a food source's trial counter exceeds the limit, abandon it
        and replace with a new random solution.
        """
        for ix in range(self.__no_of_bees):
            if self.__trials[ix] > self.__limit:
                self.__population[ix]    = self.__random_solution()
                self.__objective_vec[ix] = self.__objective_function(self.__population[ix])
                penalty                  = self.__penalty(self.__population[ix])
                self.__fitness_vec[ix]   = self.__fitness_function(
                    (self.__objective_vec[ix], penalty)
                )
                self.__trials[ix] = 0

    # ----------------------------------------------------------------
    def solution(self):
        """
        Main ABC loop:
          1. Employed Phase
          2. Onlooker Phase
          3. Elitism (save best so far)
          4. Scout Phase
        """
        best_solution_ever = None
        best_solution_obj  = None
        best_solution_fit  = None

        for iteration in range(self.__iterations):
            self.__employed_phase()
            self.__onlooker_phase()

            # --- Elitism: keep best solution alive ---
            best_index    = list(self.__fitness_vec).index(max(self.__fitness_vec))
            best_solution = self.__population[best_index]
            best_obj      = self.__objective_vec[best_index]
            best_fit      = self.__fitness_vec[best_index]

            if best_solution_fit is None or best_fit > best_solution_fit:
                best_solution_ever = best_solution.copy()
                best_solution_obj  = best_obj
                best_solution_fit  = best_fit

            self.__scout_phase()

            # Print progress every 200 iterations
            if (iteration + 1) % 200 == 0 or iteration == 0:
                print(f"\rIteration {iteration+1:>5}/{self.__iterations} | "
                      f"Best f(x) = {best_solution_obj:.4f} | "
                      f"Fitness = {best_solution_fit:.4f} | "
                      f"Total BW used = {sum(best_solution_ever):.2f} Gbps",
                      end="")

        print()  # newline after progress
        return best_solution_ever, best_solution_obj


# ------------------------------------------------------------------
# Run ABC
# ------------------------------------------------------------------

NO_OF_BEES  = 30
ITERATIONS  = 2000
LIMIT       = 25
PENALTY     = 10_000
LB          = 0.1   # each client must receive at least 0.1 Gbps

print("\n" + "=" * 65)
print("  Running ABC  |  Bees=30  Iterations=2000  Limit=25")
print("=" * 65)

problem = networking(
    users_priority = priorities,
    no_of_bees     = NO_OF_BEES,
    iterations     = ITERATIONS,
    limit          = LIMIT,
    penalty        = PENALTY,
    lb             = LB,
    ub             = TOTAL_BW
)

best_sol, best_obj = problem.solution()

# ------------------------------------------------------------------
# Results
# ------------------------------------------------------------------
print("\n" + "=" * 65)
print("                    FINAL RESULTS")
print("=" * 65)

print(f"\n{'Client':<10} {'Priority':>10} {'Max Allowed':>13} {'Allocated BW':>14} {'% of Max':>10}")
print("-" * 60)
for i in range(6):
    pct = (best_sol[i] / ub_per_client[i]) * 100
    print(f"  {i+1:<8} {priorities[i]:>10} {ub_per_client[i]:>12.1f}  {best_sol[i]:>12.2f}   {pct:>8.1f}%")

print("-" * 60)
print(f"  {'TOTAL':<8} {'':>10} {'':>13}  {sum(best_sol):>12.2f}   "
      f"{'(pool: ' + str(TOTAL_BW) + ' Gbps)':>12}")

print(f"\n  Objective f(x) = {best_obj:.6f}")
print(f"  Constraint satisfied: {sum(best_sol) <= TOTAL_BW}")

# Proportional fairness check (theoretical optimum)
sum_priorities = sum(priorities)
print("\n" + "=" * 65)
print("  Proportional Fairness Verification  (xi = B × wi / Σw)")
print("=" * 65)
print(f"\n{'Client':<10} {'Priority':>10} {'Theoretical':>13} {'ABC Result':>12} {'Diff':>8}")
print("-" * 55)
for i in range(6):
    theoretical = TOTAL_BW * priorities[i] / sum_priorities
    diff = abs(best_sol[i] - theoretical)
    print(f"  {i+1:<8} {priorities[i]:>10} {theoretical:>12.2f}  {best_sol[i]:>11.2f}  {diff:>7.2f}")


       Lab 07 – Task #3 : ABC – Bandwidth Allocation

Total Bandwidth Pool : 750 Gbps

Client     Tier/Type          Max BW   Priority
--------------------------------------------------
  1        2/Special          130.0          3
  2        3/Regular           32.5          2
  3        1/Special          260.0          4
  4        4/Regular           10.0          1
  5        1/Special          200.0          4
  6        2/Special          122.0          3


  Running ABC  |  Bees=30  Iterations=2000  Limit=25
Iteration  2000/2000 | Best f(x) = 81.6458 | Fitness = 81.6458 | Total BW used = 749.99 Gbps

                    FINAL RESULTS

Client       Priority   Max Allowed   Allocated BW   % of Max
------------------------------------------------------------
  1                 3        130.0        130.00      100.0%
  2                 2         32.5         32.50      100.0%
  3                 4        260.0        255.49       98.3%
  4                 1         10.0        